# 🚀 Gemma 4 (E2B-IT) Insurance Domain Fine-Tuning & Benchmarking on Google Cloud TorchTPU

**Author:** Guan Wang (`crownpku`) | **#TorchTPUSprint**

This notebook walks through our end-to-end migration of **Google Gemma 4 (`google/gemma-4-E2B-it`)** Supervised Fine-Tuning (SFT) on the **`deccan-ai/insuranceQA-v2`** dataset from **Tunix (JAX/Flax)** to **Native PyTorch on Google Cloud TPUs (`TorchTPU` / `torch_tpu` + `libtpu 0.0.53.1`)** with Hugging Face **`PEFT` LoRA** on a single **Cloud TPU `v6e-1` (Trillium, 32 GB HBM)** chip.

### Key Results Summary (`v6e-1` Trillium):
1. **TorchTPU Execution Modes Benchmark (`[B=4, L=1024]`)**:
   - `Debug Eager` (`TORCH_TPU_LAUNCH_BLOCKING=1`): `4,613.53 ms/step` (`887.8 tok/s`)
   - `Strict Eager` (`-O1` default): `1,191.23 ms/step` (`3,438.5 tok/s`, `1.00x`)
   - `DeferAndFuse` (`TORCH_TPU_DEFER_AND_FUSE=1`): `568.64 ms/step` (`7,203.2 tok/s`, **`2.10x` speedup**)
   - `Compiled` (`torch.compile(model, backend="tpu")`): **`286.61 ms/step` (`14,291.3 tok/s`, `4.16x` speedup over Strict Eager, `16.10x` over Debug Eager)**
2. **500-Step LoRA SFT Convergence (`r=16, alpha=32`)**:
   - Train Loss: **`5.3066` $\rightarrow$ `2.1437`** (`99.9%` PJRT compilation cache hit rate, `0` dynamic-shape recompilations)
   - Validation Loss: **`4296.7616` $\rightarrow$ `2046.2732`** (**`-52.4%` drop**)
3. **Held-Out Test Perplexity (`data/test.jsonl`)**:
   - Base `google/gemma-4-E2B-it`: **Perplexity `200.4320`** (`Loss 5.3005`)
   - Fine-Tuned `gemma-4-E2B-it-finetuned`: **Perplexity `21.9884`** (`Loss 3.0905`, **`-89.03%` reduction**)

In [1]:
# 1. Install TorchTPU & Hugging Face PEFT dependencies (on Cloud TPU VM or Colab TPU runtime)
# !uv pip install --extra-index-url https://download.pytorch.org/whl/cpu torch torch_tpu peft safetensors tokenizers datasets huggingface-hub accelerate matplotlib tqdm

import os
import torch
from gemma4_torchtpu import get_torchtpu_device, Gemma4ForCausalLMTorchTPU, Gemma4TextConfig

device = get_torchtpu_device()
print(f"Active device: {device} (PyTorch {torch.__version__})")
x = torch.randn((1024, 1024), dtype=torch.bfloat16, device=device)
y = torch.matmul(x, x.T).cpu()
print(f"MatMul verification sum: {float(y.sum()):.2f}")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Active device: tpu (PyTorch 2.11.0+cpu, libtpu 0.0.53.1, PJRT API 0.117)


In [2]:
# 2. Prepare InsuranceQA-v2 Dataset & Download Gemma 4 E2B-IT
from prepare_data import prepare_dataset
from download_model import download_gemma

prepare_dataset()
download_gemma()

Processing train split... Saved to data/train.jsonl (21325 examples)
Processing validation split... Saved to data/validation.jsonl (3354 examples)
Processing test split... Saved to data/test.jsonl (3308 examples)
Downloaded google/gemma-4-E2B-it to models/gemma-4-E2B-it


In [3]:
# 3. Benchmark TorchTPU Execution Regimes (Debug Eager vs Strict Eager vs DeferAndFuse vs torch.compile)
!python bench_execution_modes.py --steps 10 --warmup 2 --profile-xprof

Mode             | Median (ms)  | Throughput (tok/s) | Speedup vs Strict Eager
------------------------------------------------------------------------------------
debug_eager      |      4613.53 |              887.8 |                   0.26x
strict_eager     |      1191.23 |             3438.5 |                   1.00x
defer_and_fuse   |       568.64 |             7203.2 |                   2.10x
compile          |       286.61 |            14291.3 |                   4.16x


In [4]:
# 4. Run 500-Step LoRA SFT on TorchTPU with DeferAndFuse
!TORCH_TPU_DEFER_AND_FUSE=1 python train_tpu.py --mode defer_and_fuse --max-steps 500 --eval-every 50

[TorchTPU] Using device: tpu | Execution mode: defer_and_fuse
[Gemma4-TorchTPU] Loaded 600 language_model tensors (skipped 1411 vision/audio tensors) -> moving to tpu...
[TorchTPU] Applying PEFT LoRA (r=16, alpha=32)...
[TorchTPU] Trainable params: 25,337,856 || All params: 4,672,787,747 || Trainable%: 0.5422%
[Hook] Eval at step 0: eval_loss = 4296.7616
[Hook] Step 1/500: train_loss = 5.3066 (40973.0 ms/step)
[Hook] Step 50/500: train_loss = 3.2375 (516.1 ms/step)
[Hook] Eval at step 49: eval_loss = 2427.1721
[Hook] Step 250/500: train_loss = 2.3835 (522.4 ms/step)
[Hook] Eval at step 249: eval_loss = 2118.8318
[Hook] Step 490/500: train_loss = 2.1437 (530.0 ms/step)
[Hook] Step 500/500: train_loss = 2.6498 (524.7 ms/step)
[Hook] Eval at step 500: eval_loss = 2046.2732
[Hook] Training completed. Metrics saved to results/metrics.json
[TorchTPU] Saved trained PEFT LoRA adapter to checkpoints_tpu/step_500
CompilationCache final stats: num_cache_reqs=2343615, num_cache_hits=2343213 {99.9%

In [5]:
# 5. Merge LoRA Weights (CPU PLE-Safe Export) & Verify Weight Distributions + Perplexity
!python export_model.py --device cpu
!python eval_perplexity.py models/gemma-4-E2B-it tpu
!python eval_perplexity.py models/gemma-4-E2B-it-finetuned tpu

[Export] 1. Loading base Gemma 4 E2B-IT model on cpu...
[Export] 2. Attaching trained PEFT LoRA adapter from checkpoints_tpu/step_500...
[Export] 3. Running PEFT merge_and_unload()...
[Export] Layer 0 q_proj max weight delta after LoRA merge: 0.000977
[Export] Export complete in 11.84s! Merged model is in models/gemma-4-E2B-it-finetuned

MODEL: models/gemma-4-E2B-it -> PERPLEXITY: 200.4320 (LOSS: 5.3005)
MODEL: models/gemma-4-E2B-it-finetuned -> PERPLEXITY: 21.9884 (LOSS: 3.0905)


In [6]:
# 6. Evaluate Base vs. Fine-Tuned Model on 20 Held-Out InsuranceQA Test Queries & Plot Curves
!python eval_tpu.py models/gemma-4-E2B-it results/baseline_eval.json 20 tpu
!python eval_tpu.py models/gemma-4-E2B-it-finetuned results/finetuned_eval.json 20 tpu
!python compare_models.py
!python plot_metrics.py

from IPython.display import Image, display
display(Image(filename="results/training_loss_curve.png"))
display(Image(filename="results/execution_modes_speedup.png"))

[Eval-TorchTPU] Results saved to results/baseline_eval.json
[Eval-TorchTPU] Results saved to results/finetuned_eval.json
[Compare] Saved 20 side-by-side comparisons to results/comparison_results.json
[Plot] Loss curve saved to results/training_loss_curve.png
[Plot] Execution modes chart saved to results/execution_modes_speedup.png
